In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-675f97c0-7f98-6b94-50a5-32df8d247c37)


In [2]:
# Upgrade pip and install the correct package names
!pip install --upgrade pip
!pip install -q kagglehub ultralytics scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [3]:
import os
import shutil
import kagglehub
from sklearn.model_selection import train_test_split

# 1. Download dataset
path = kagglehub.dataset_download("lylmsc/wider-face-for-yolo-training")
print("Dataset downloaded to:", path)

# 2. Define source paths (Kagglehub usually puts them in a subfolder)
# Note: Adjust 'wider_face' if the folder name inside 'path' differs
src_images = os.path.join(path, 'images')
src_labels = os.path.join(path, 'labels')

# 3. Define target root directory in Colab
root_dir = '/content/datasets/wider_face'
os.makedirs(root_dir, exist_ok=True)

# 4. Get filenames
all_images = [os.path.splitext(f)[0] for f in os.listdir(src_images) if f.endswith('.jpg')]

# 5. Split data (80/10/10)
train_files, temp_files = train_test_split(all_images, train_size=0.8, random_state=42)
val_files, test_files = train_test_split(temp_files, train_size=0.5, random_state=42)

# 6. Create folders and move files
sets = {'train': train_files, 'val': val_files, 'test': test_files}

for set_name, files in sets.items():
    dest_img_dir = os.path.join(root_dir, set_name, 'images')
    dest_lbl_dir = os.path.join(root_dir, set_name, 'labels')
    os.makedirs(dest_img_dir, exist_ok=True)
    os.makedirs(dest_lbl_dir, exist_ok=True)

    for f in files:
        # Copy instead of move to keep the original download intact
        shutil.copy(os.path.join(src_images, f + '.jpg'), os.path.join(dest_img_dir, f + '.jpg'))
        shutil.copy(os.path.join(src_labels, f + '.txt'), os.path.join(dest_lbl_dir, f + '.txt'))

print("Dataset organized in /content/datasets/wider_face")

100%|██████████| 2.45G/2.45G [01:51<00:00, 23.5MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/lylmsc/wider-face-for-yolo-training/versions/1
Dataset organized in /content/datasets/wider_face


In [4]:
yaml_content = f"""
path: {root_dir}
train: train/images
val: val/images
test: test/images

names:
  0: face
"""

with open('data.yaml', 'w') as f:
    f.write(yaml_content)

In [ ]:
from ultralytics import YOLO
from datetime import datetime
import torch

# 1. Verify GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 2. Load model
model = YOLO("yolo26n.pt")

# 3. Optimized Train call for T4
results = model.train(
    data="data.yaml",
    epochs=10,
    imgsz=640,
    batch=32,            # dont use 32 becuase the T4 gpu runs our of memory
    device=0,            # Explicitly use the first NVIDIA GPU
    workers=2,            # Colab free tier often throttles if workers > 2
    cache='ram',         # Uses Colab's high-speed system RAM to store images
    exist_ok=True,       # Overwrite existing experiment folders if needed
    pretrained=True,     # Ensure transfer learning is active
    optimizer='auto',    # Let YOLO choose between MuSGD or AdamW based on your data
    amp=True,            # Enable Automatic Mixed Precision (essential for T4 speed)
    project='face_detection', # Organizes your runs
    name='yolo26n_t4_run'
)

# 4. Save and export
timestamp = datetime.now().strftime("%d%b_%H%M")
save_path = f"yolo26n_face_{timestamp}.pt"
model.save(save_path)

# Optional: Export to ONNX for faster inference later
# model.export(format="onnx")

print(f"\nModel successfully saved to: {save_path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=Fals